*Imports and hepfunctions*

In [1]:
#Interactive library
import plotly.express as px

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim

# String matching
from rapidfuzz import process

import re


In [2]:
def normalize_title(title):
    title = title.lower()
    title = title.replace(" ", "")
    title = re.sub(r"\(\d{4}\)", "", title)   # Source: See References [2] below
    return title

*Load data*

In [3]:
# Source: See References [1] below
directory_data = "../data" # Adjust the path to your data directory if necessary    
movies = pd.read_csv(f"{directory_data}/raw/movies.csv")  # Adjust the path to your data directory if necessary
ratings = pd.read_csv(f"{directory_data}/raw/ratings.csv")
tags = pd.read_csv(f"{directory_data}/raw/tags.csv")

*tags per movie*

*tags per movie*

In [4]:
# Source: See References [3] below

tags_per_movie = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index()
    .rename(columns={"tag": "tags_combined"})
)

*merge tags and create textfeatures*

In [5]:
movies = movies.merge(tags_per_movie, on="movieId", how="left")

#fill missing tags
movies["tags_combined"] = movies["tags_combined"].fillna("")

In [6]:

#clean title
movies["clean_title"] = movies["title"].apply(normalize_title)

#create new content column with genres and tags:            Source: See References [4] below
movies["genres_clean"] = (
    movies["genres"]
    .str.replace("(no genres listed)", "", regex=False)
    .str.replace("|", " ", regex=False)
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["tags_clean"] = (
    movies["tags_combined"]
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["content_with_tags"] = (
    movies["genres_clean"]
    + " "
    + movies["tags_clean"].str.replace(r"\b\d+\b", 
    "", 
    regex=True
    )
)

movies["content_with_tags"] = (
    movies["content_with_tags"]
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)   # Only keep latin letters and spaces - chatGPT for regex
    .str.replace(r"\s+", " ", regex=True)          # Replace multiple spaces with a single space
    .str.strip()                                   # Remove leading and trailing spaces
)



*TF IDF vectorization*

In [7]:
#  TF-IDF vectorization on combined genres + tags
tfidf_tags = TfidfVectorizer(stop_words="english")
tfidf_matrix_tags = tfidf_tags.fit_transform(movies["content_with_tags"])
feature_names = tfidf_tags.get_feature_names_out()

*Create wighting rating*  

In [8]:

# Source: See References [5] below

user_counts = ratings.groupby("userId").size().reset_index(name="user_rating_count")

#merge in ratings
ratings = ratings.merge(user_counts, on="userId", how="left") 

#create userweight
ratings["user_weight"] = 1 / np.log1p(ratings["user_rating_count"])

# filter out users with less than 5 ratings
ratings = ratings[ratings["user_rating_count"] > 5]

#create weighted rating per row
ratings["weighted_user_rating"] = ratings["rating"] * ratings["user_weight"]



*create ratings summary per movie*

In [9]:
ratings_summary = (
    ratings.groupby("movieId")
    .agg(
        weighted_sum = ("weighted_user_rating", "sum"),
        weight_total = ("user_weight", "sum"),
        rating_count = ("rating", "count")
    )
    .reset_index()
)

ratings_summary["average_rating"] = (
    ratings_summary["weighted_sum"] / ratings_summary["weight_total"]
)

https://code.energy/write-code-rates-fairly/  about imdb ratings

*Merge ratings into movie*

In [10]:
movies = movies.drop(
    columns = ["average_rating", "rating_count", "weighted_rating"],
    errors = "ignore"
)

movies = movies.merge(
    ratings_summary[["movieId", "average_rating", "rating_count" ]],
    on= "movieId",
    how = "left"
)

movies["average_rating"] = movies["average_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0)


*Create column weighted rating*  
[6], [7]

weighted_rating = v/(v+m)R+m/(v+m)C  

R = average_rating  
V = rating_count  
c = mean_rating in the whole data  
m = a threshold for how many ratings are required before we can trust the movie’s own average more (how many is "a lot" of ratings)

In [11]:
c = movies["average_rating"].mean()

m = movies["rating_count"].quantile(0.90)

movies["weighted_rating"] = (
    (movies["rating_count"] / (movies["rating_count"] + m)) * movies["average_rating"]
    + (m / (movies["rating_count"] + m)) * c
)

**Final recommendation function**

**final recommendation function**

In [20]:
def recommend_movies_with_reranking(movie_title, n=5, movie_pool=20):
    
    # find matching movies
    matches = movies[movies["title"].str.contains(
        movie_title,
        case=False,
        na=False,
        regex=False
    )]

    if len(matches) == 0:
        print("Movie could not be found")
        return None
    
    if len(matches) > 1:
        print("Multiple movies found. Using the first match: ")
        print(matches[["title"]].head(3))    

    movie_index = matches.index[0]

    title = movies.loc[movie_index, "title"]
    
    # calculate similarity
    similarity_score = cosine_sim(
        tfidf_matrix_tags[movie_index],
        tfidf_matrix_tags
    ).flatten()

    # get most similar movies
    sorted_movie_indexes = similarity_score.argsort()[::-1]
    movie_index_no_input = [movie_idx for movie_idx in sorted_movie_indexes if movie_idx != movie_index]

    candidate_indexes = movie_index_no_input[:movie_pool]

    # create result dataframe
    result = movies.iloc[candidate_indexes][
        [
            "title",
            "genres",
            "tags_combined",
            "average_rating",
            "rating_count",
            "weighted_rating"
        ]
    ].copy()

    # add similarity score to the result df
    result["similarity_score"] = similarity_score[candidate_indexes]

    # normalize rating to 0–1 safely
    min_rating = result["weighted_rating"].min()
    max_rating = result["weighted_rating"].max()

    if max_rating == min_rating:
        result["weighted_rating_norm"] = 0
    else:
        result["weighted_rating_norm"] = (
            (result["weighted_rating"] - min_rating) /
            (max_rating - min_rating)
        )

    # combine similarity, rating and popularity 
    # [8]
    result["final_score"] = (
        0.7 * result["similarity_score"] +
        0.2 * result["weighted_rating_norm"] +
        0.1 * np.log1p(result["rating_count"])
    )

    # sort by final score
    result = result.sort_values("final_score", ascending=False)

    # apply threshold to separate "safe" and "hidden" recommendations
    threshold = 100

    safe = result[result["rating_count"] >= threshold].head(3)
    hidden = result[result["rating_count"] < threshold].head(2)

    print("safe:", len(safe))
    print("hidden:", len(hidden))

    # combine safe and hidden
    result = pd.concat([safe, hidden])

    # remove duplicate titles
    already_seen = set()
    unique_results = []

    for _, row in result.iterrows():
        name = row["title"].split("(")[0].strip().lower()
        
        if name not in already_seen:
            unique_results.append(row)
            already_seen.add(name)

        if len(unique_results) == n:
            break

    result = pd.DataFrame(unique_results)
    result = result.sort_values("final_score", ascending=False)

    print(f"Recommendations for '{title}' after reranking:")

    return result

**testblock**

In [21]:
recommend_movies_with_reranking("Toy Story", n=5, movie_pool=20)


Multiple movies found. Using the first match: 
                    title
0        Toy Story (1995)
3021   Toy Story 2 (1999)
14815  Toy Story 3 (2010)
safe: 3
hidden: 2
Recommendations for 'Toy Story (1995)' after reranking:


,title,genres,tags_combined,average_rating,rating_count,weighted_rating,similarity_score,weighted_rating_norm,final_score
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,whimsical cgi Disney Pixar very good but the f...,3.820162,34440.0,3.813854,0.900382,0.863198,1.847607
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,cute funny innovative Disney Pixar cute funny ...,3.854475,48389.0,3.849806,0.718433,0.893162,1.760240
6259,Finding Nemo (2003),Adventure|Animation|Children|Comedy,Comedy family bonds father-son relationship fu...,3.834102,48058.0,3.829503,0.707338,0.876240,1.748403
26652,The Adventures of André and Wally B. (1984),Animation,computer animation pixar animation,2.159707,54.0,2.778160,0.673182,0.000000,0.871961
21340,George & A.J. (2009),Animation|Children|Comedy,memasa's movies Pixar short pixar animation short,2.493148,34.0,2.864052,0.636512,0.071586,0.815411


For Toy Story (1995), the model produced relevant and diverse recommendations. Instead of returning mostly sequels, it included other Pixar movies with similar themes and genres. This shows that the combination of content-based similarity and reranking improved both relevance and diversity.

In [ ]:
recommend_movies_with_reranking("Back to the Future (1985)", n=5, candidates_for_reranking=20)

safe: 2
hidden: 2
Recommendations for 'Back to the Future (1985)' after reranking:


,title,genres,tags_combined,average_rating,rating_count,weighted_rating,similarity_score,weighted_rating_norm,final_score
1922,Back to the Future Part II (1989),Adventure|Comedy|Sci-Fi,Christopher Lloyd classic cliffhanger time tra...,3.584428,29648.0,3.579017,0.822064,1.000000,1.805163
1923,Back to the Future Part III (1990),Adventure|Comedy|Sci-Fi|Western,Alan Silvestri Christopher Lloyd Michael J. Fo...,3.350510,28018.0,3.346791,0.733213,0.653586,1.668030
64986,Serf (2019),Comedy,time travel,3.303207,41.0,2.972236,0.654199,0.094860,0.850678
50675,Nothing Left to Do but Cry (1984),Comedy,time travel,3.808812,24.0,2.996702,0.654199,0.131357,0.806098


For Back to the Future, the reranked recommendations were partly strong and partly weaker. The top results, Back to the Future Part II and Part III, were highly relevant due to both strong similarity and high rating support. However, some lower-ranked recommendations were mainly connected through the tag time travel, which shows that the model can identify meaningful thematic overlap but may still return less intuitively related movies when the candidate pool is limited.

## References
  
[1] Regex pattern generated by ChatGPT: "How do I remove the year from the movie title?"
  
[2] Data loading approach adapted from: [Laura_clean.ipynb](https://github.com/pontuskungsbacka/Project_OS_Australia/blob/main/notebooks/Laura_clean.ipynb)

[3] chatGPT: “How can I group rows in and combine text values into one string?”

[4] chatGPT: "I do not remenber how to group rows and combine text values into one string per group"

[5] chatgGPT: How can I weight user ratings to avoid bias from very active users?"

[6] https://code.energy/write-code-rates-fairly/

[7] chatGPT: “Hur undviker jag att filmer med få men höga betyg hamnar högst?”, “Hur väljer jag ett rimligt värde för m?”

[8] chatGPT: "Hur kan jag kombinera similarity med ratings?”, "Hur väljer jag hur stor inverkan de har på modellen?", “Hur jämför jag före och efter?”, “är detta imdb:s bayesian rating?”

